# Kaggle Word2Vec NLP Tutorial Part 2：Word Vectors

这一部分开始训练 Word2Vec。Part 1 的词袋模型只统计词频，Part 2 要把每个词训练成一个稠密向量。

Word2Vec 不需要情感标签，所以这里会使用 `labeledTrainData.tsv` 和 `unlabeledTrainData.tsv`。未标注评论虽然不能直接训练分类器，但可以帮助模型学习词语之间的上下文关系。


## 1. 导入库

这部分比 Part 1 多用到 `gensim`。Kaggle Notebook 一般已经预装；如果本地运行，需要先安装 `gensim`。


In [ ]:
import csv
import re
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
from bs4 import BeautifulSoup
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

from gensim.models import Word2Vec
import logging



## 2. 查找 Kaggle 数据路径

这个单元格会自动查找 Kaggle 的输入目录，也支持本地 `data/` 目录。


In [ ]:
def show_available_inputs():
    input_root = Path('/kaggle/input')
    if input_root.exists():
        print('Available /kaggle/input entries:')
        for path in sorted(input_root.glob('*')):
            print(' -', path)
            for child in sorted(path.glob('*'))[:10]:
                print('    -', child.name)
    else:
        print('/kaggle/input does not exist. If running locally, put files under data/.')

def find_file(filename, search_dirs, required=True):
    for directory in search_dirs:
        directory = Path(directory)
        direct_path = directory / filename
        zipped_path = directory / f'{filename}.zip'
        if direct_path.exists():
            return direct_path
        if zipped_path.exists():
            return zipped_path
        if directory.exists():
            matches = list(directory.rglob(filename))
            if matches:
                return matches[0]
            zipped_matches = list(directory.rglob(f'{filename}.zip'))
            if zipped_matches:
                return zipped_matches[0]
    if required:
        show_available_inputs()
        raise FileNotFoundError(
            f'Cannot find {filename}. In Kaggle, click Add Input / Add data and attach the word2vec-nlp-tutorial competition dataset.'
        )
    return None

def read_tsv(path):
    path = Path(path)
    if path.suffix == '.zip':
        with zipfile.ZipFile(path) as archive:
            tsv_names = [name for name in archive.namelist() if name.endswith('.tsv')]
            if not tsv_names:
                raise ValueError(f'No TSV file found inside {path}')
            with archive.open(tsv_names[0]) as file_obj:
                return pd.read_csv(file_obj, header=0, delimiter='\t', quoting=csv.QUOTE_NONE)
    return pd.read_csv(path, header=0, delimiter='\t', quoting=csv.QUOTE_NONE)

search_dirs = [
    '/kaggle/input/word2vec-nlp-tutorial',
    '/kaggle/input',
    'data',
    '.',
]

train_path = find_file('labeledTrainData.tsv', search_dirs)
test_path = find_file('testData.tsv', search_dirs, required=False)
unlabeled_path = find_file('unlabeledTrainData.tsv', search_dirs)

print('Train file:    ', train_path)
print('Test file:     ', test_path)
print('Unlabeled file:', unlabeled_path)


## 3. 读取训练集、测试集和未标注训练集

Part 2 的重点是 `unlabeledTrainData.tsv`。它没有 `sentiment`，但有大量评论文本，可以用于 Word2Vec 的无监督训练。


In [ ]:
train = read_tsv(train_path)
unlabeled_train = read_tsv(unlabeled_path)
test = read_tsv(test_path) if test_path is not None else None

print('Train shape:    ', train.shape)
print('Unlabeled shape:', unlabeled_train.shape)
if test is not None:
    print('Test shape:     ', test.shape)
    print(f"Read {train['review'].size} labeled train reviews, {test['review'].size} test reviews, and {unlabeled_train['review'].size} unlabeled reviews.")
else:
    print(f"Read {train['review'].size} labeled train reviews and {unlabeled_train['review'].size} unlabeled reviews.")

display(train.head())



## 4. 文本清洗和切句

Word2Vec 的输入是一组句子。每个句子是一个单词列表。

这里默认不去掉停用词，因为 Word2Vec 依赖上下文，`the`、`is`、`of` 这类词虽然对情感分类帮助不大，但它们能帮助模型理解句子结构。


In [ ]:
STOP_WORDS = set(ENGLISH_STOP_WORDS)

def review_to_wordlist(raw_review, remove_stopwords=False):
    text = BeautifulSoup(raw_review, 'html.parser').get_text()
    text = re.sub('[^a-zA-Z]', ' ', text)
    words = text.lower().split()
    if remove_stopwords:
        words = [word for word in words if word not in STOP_WORDS]
    return words

def simple_sentence_split(raw_review):
    return [sentence.strip() for sentence in re.split(r'[.!?]+', raw_review) if sentence.strip()]

try:
    import nltk.data
    tokenizer = nltk.data.load('tokenizers/punkt/english.pickle')
    print('Using NLTK punkt sentence tokenizer.')
except Exception:
    tokenizer = None
    print('NLTK punkt tokenizer is not available. Falling back to simple regex sentence splitting.')

def review_to_sentences(raw_review, remove_stopwords=False):
    if tokenizer is not None:
        raw_sentences = tokenizer.tokenize(raw_review.strip())
    else:
        raw_sentences = simple_sentence_split(raw_review)

    sentences = []
    for raw_sentence in raw_sentences:
        words = review_to_wordlist(raw_sentence, remove_stopwords=remove_stopwords)
        if words:
            sentences.append(words)
    return sentences


## 5. 准备 Word2Vec 训练语料

原教程大约会得到 85 万个句子。这个过程可能需要几分钟。

如果只是快速演示，可以把 `MAX_REVIEWS_FOR_DEMO` 改成一个数字，比如 `5000`。正式运行建议保持 `None`。


In [ ]:
MAX_REVIEWS_FOR_DEMO = None

sentences = []

train_reviews = train['review'] if MAX_REVIEWS_FOR_DEMO is None else train['review'].head(MAX_REVIEWS_FOR_DEMO)
unlabeled_reviews = unlabeled_train['review'] if MAX_REVIEWS_FOR_DEMO is None else unlabeled_train['review'].head(MAX_REVIEWS_FOR_DEMO)

print('Parsing sentences from labeled training set...')
for i, review in enumerate(train_reviews, start=1):
    if i % 5000 == 0:
        print(f'Parsed {i} labeled reviews')
    sentences.extend(review_to_sentences(review, remove_stopwords=False))

print('Parsing sentences from unlabeled training set...')
for i, review in enumerate(unlabeled_reviews, start=1):
    if i % 5000 == 0:
        print(f'Parsed {i} unlabeled reviews')
    sentences.extend(review_to_sentences(review, remove_stopwords=False))

print('Total sentences:', len(sentences))
print('Example sentence:', sentences[0][:30])


## 6. 训练 Word2Vec 模型

几个核心参数：

`vector_size=300` 表示每个词用 300 维向量表示。

`window=10` 表示用当前词前后最多 10 个词作为上下文。

`min_count=40` 表示出现少于 40 次的词不进入词表。

`sample=1e-3` 用来降低高频词的影响。

`workers=4` 表示并行训练。Kaggle CPU 就能跑，不需要 GPU。


In [ ]:
logging.basicConfig(format='%(asctime)s : %(levelname)s : %(message)s', level=logging.INFO)

NUM_FEATURES = 300
MIN_WORD_COUNT = 40
NUM_WORKERS = 4
CONTEXT = 10
DOWNSAMPLING = 1e-3

model = Word2Vec(
    sentences=sentences,
    vector_size=NUM_FEATURES,
    min_count=MIN_WORD_COUNT,
    workers=NUM_WORKERS,
    window=CONTEXT,
    sample=DOWNSAMPLING,
    sg=0,
    seed=42,
)

print('Vocabulary size:', len(model.wv.index_to_key))
print('Vector size:', model.wv.vector_size)



## 7. 保存模型

Part 3 会继续使用这个模型。Kaggle 运行时建议保存到 `/kaggle/working/`。


In [ ]:
output_dir = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('.')
model_path = output_dir / '300features_40minwords_10context.model'
model.save(str(model_path))
print('Saved Word2Vec model:', model_path)


## 8. 查看词向量效果

`most_similar` 会寻找向量空间里最接近的词。

`doesnt_match` 会从一组词中找出最不合群的那个词。


In [ ]:
def show_if_in_vocab(word):
    if word in model.wv:
        print(f'Words similar to {word}:')
        for similar_word, score in model.wv.most_similar(word)[:10]:
            print(f'{similar_word:15s} {score:.4f}')
    else:
        print(f'{word!r} is not in vocabulary.')

show_if_in_vocab('awful')
print()
show_if_in_vocab('good')

candidates = [word for word in ['man', 'woman', 'child', 'kitchen'] if word in model.wv]
if len(candidates) >= 3:
    print('\nDoes not match:', model.wv.doesnt_match(candidates))


## 9. 小结

Part 2 的重点不是直接分类，而是训练词向量。

词袋模型把词当成互相独立的符号，Word2Vec 会根据上下文把词映射到连续向量空间。语义相近的词通常会有更接近的向量。

下一步 Part 3 要解决的问题是：已经有了词向量，怎样把一整篇影评变成可以输入分类器的特征。
